# Задание 1. Реализация PWM "с нуля"

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [4]:
sites = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC",
         "CAGGTTGGC"]
ncltd = ['A', 'T', 'G', 'C'] #выпишем все нуклеотиды 

In [5]:
def pfm(sites, nucleotids):
    seq_length = len(sites[0]) #находим длину последовательности - просто возьмем длину нулевой - они все одинаковые
    n_sites = len(sites) #находим количество последовательностей всего - чтобы дальше для каждой посчитать 
    
    pfm_matrix = np.zeros((4, seq_length), dtype=int) #создаем матрицу необходимый размеров 4*n
    for i in range(seq_length):
        for nuc_indx, nuc in enumerate(nucleotids):
            count = 0 
            for site in sites:
                if site[i] == nuc:
                    count += 1
            pfm_matrix[nuc_indx][i] = count
    return pfm_matrix, ncltd

In [6]:
pfm_result, nucleotides = pfm(sites, ncltd)

df = pd.DataFrame(pfm_result, 
                  index=nucleotides, 
                  columns=[i for i in range(len(sites[0]))])

print(df)

   0  1  2   3   4  5  6  7  8
A  1  8  1   0   0  2  7  2  1
T  2  0  0   0  10  2  0  0  1
G  1  0  8  10   0  0  3  8  0
C  6  2  1   0   0  6  0  0  8


In [7]:
def pfm_to_ppm(pfm_matrix, alpha):
    total_sites = np.sum(pfm_matrix[:, 0]) 
    num_ncltd, seq_length = pfm_matrix.shape
    ppm_matrix = np.zeros((4, seq_length), dtype=float)
    for i in range (num_ncltd):
        for j in range(seq_length):
            ppm_matrix[i][j] = (pfm_matrix[i][j] + alpha)/(total_sites + 4 * alpha)
    return ppm_matrix

In [8]:
ppm_result = pfm_to_ppm(pfm_result, 0.1)

df_ppm = pd.DataFrame(ppm_result, 
                  index=nucleotides, 
                  columns=[i for i in range(len(sites[0]))])

print(df_ppm)

          0         1         2         3         4         5         6  \
A  0.105769  0.778846  0.105769  0.009615  0.009615  0.201923  0.682692   
T  0.201923  0.009615  0.009615  0.009615  0.971154  0.201923  0.009615   
G  0.105769  0.009615  0.778846  0.971154  0.009615  0.009615  0.298077   
C  0.586538  0.201923  0.105769  0.009615  0.009615  0.586538  0.009615   

          7         8  
A  0.201923  0.105769  
T  0.009615  0.105769  
G  0.778846  0.009615  
C  0.009615  0.778846  


In [9]:
def ppm_to_pwm(ppm_matrix, nucleotides, p_a, p_t, p_g, p_c):
    num_ncltd, seq_length = ppm_matrix.shape
    pwm_matrix = np.zeros((4, seq_length), dtype=float)
    for i in range(seq_length):
        for nuc_indx, nuc in enumerate(nucleotides):
            if nuc == 'A':
               pwm_matrix[nuc_indx][i] =  math.log2(ppm_matrix[nuc_indx][i]/p_a)
            elif nuc == 'T':
               pwm_matrix[nuc_indx][i] =  math.log2(ppm_matrix[nuc_indx][i]/p_t)
            elif nuc == 'G':
                pwm_matrix[nuc_indx][i] =  math.log2(ppm_matrix[nuc_indx][i]/p_g)
            elif nuc == 'C':
                pwm_matrix[nuc_indx][i] =  math.log2(ppm_matrix[nuc_indx][i]/p_c)
    return pwm_matrix

In [10]:
pwm_result = ppm_to_pwm(ppm_result, ncltd, 0.295, 0.295, 0.205, 0.205)

df_pwm = pd.DataFrame(pwm_result, 
                  index=nucleotides, 
                  columns=[i for i in range(len(sites[0]))])

print(df_pwm)

          0         1         2         3         4         5         6  \
A -1.479795  1.400623 -1.479795 -4.939227 -4.939227 -0.546909  1.210521   
T -0.546909 -4.939227 -4.939227 -4.939227  1.718985 -0.546909 -4.939227   
G -0.954704 -4.414136  1.925714  2.244076 -4.414136 -4.414136  0.540061   
C  1.516602 -0.021818 -0.954704 -4.414136 -4.414136  1.516602 -4.414136   

          7         8  
A -0.546909 -1.479795  
T -4.939227 -1.479795  
G  1.925714 -4.414136  
C -4.414136  1.925714  


In [11]:
# Находим теоретические максимум и минимум
theoretical_max = 0
theoretical_min = 0
max_sequence = ""
min_sequence = ""

# Проходим по каждому столбцу (позиции)
for i in range(pwm_result.shape[1]):  # shape[1] - количество столбцов
    # Для максимума
    max_val = np.max(pwm_result[:, i])
    max_nuc_idx = np.argmax(pwm_result[:, i])
    max_nuc = ncltd[max_nuc_idx]
    
    # Для минимума  
    min_val = np.min(pwm_result[:, i])
    min_nuc_idx = np.argmin(pwm_result[:, i])
    min_nuc = ncltd[min_nuc_idx]
    
    # Суммируем
    theoretical_max += max_val
    theoretical_min += min_val
    
    # Собираем последовательности
    max_sequence += max_nuc
    min_sequence += min_nuc

print(f"Теоретический максимальный скор: {theoretical_max:.4f}")
print(f"Последовательность для максимума: {max_sequence}")
print()
print(f"Теоретический минимальный скор: {theoretical_min:.4f}")  
print(f"Последовательность для минимума: {min_sequence}")

Теоретический максимальный скор: 15.3846
Последовательность для максимума: CAGGTCAGC

Теоретический минимальный скор: -39.9434
Последовательность для минимума: ATTAAGTTG
